# 🧠 Lasmoid — Gemma-4-12B → Lasmoid-100M Knowledge Distillation

**Platform**: Kaggle T4 x2 (32GB VRAM) — FREE  
**Teacher**: `google/gemma-4-12B` (12B params, loaded in 4-bit NF4 → ~6GB VRAM)  
**Student**: Lasmoid-100M (Custom Hybrid Transformer, ~1GB VRAM)  
**Method**: Online KL-divergence distillation with progressive alpha scheduling  
**Dataset**: FineWeb-Edu (streaming, high-quality educational web text)  

---

### ⚡ Memory Budget (T4 x2 = 32GB)
| Component | VRAM |
|-----------|------|
| Gemma-4-12B (4-bit NF4) on GPU 0 | ~6.5 GB |
| Lasmoid-100M (bf16) on GPU 1 | ~1.0 GB |
| Activations + teacher logits | ~4.0 GB |
| Optimizer states | ~2.0 GB |
| **Total** | **~13.5 GB** ✅ |

### 📋 Prerequisites
1. Add your HuggingFace token as a Kaggle Secret named `HF_TOKEN`
2. Accept Gemma-4-12B terms at: https://huggingface.co/google/gemma-4-12B
3. Enable **T4 x2 GPU** in: Notebook Settings → Accelerator

## 📦 Step 1 — Install Dependencies & Check GPU

In [ ]:
import subprocess, sys

# Install required packages
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "bitsandbytes>=0.43.0",
    "transformers>=4.51.0",
    "datasets",
    "accelerate",
    "safetensors",
    "huggingface_hub",
    "tqdm"
], check=True)

import torch
print("="*60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory/1e9:.1f} GB)")
print("="*60)

assert torch.cuda.is_available(), "❌ No GPU! Enable T4 x2 in Notebook Settings."
assert torch.cuda.device_count() >= 1, "❌ Need at least 1 GPU."

# Device assignment
TEACHER_DEVICE = "cuda:0"
STUDENT_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
print(f"\n✅ Teacher → {TEACHER_DEVICE}")
print(f"✅ Student → {STUDENT_DEVICE}")

## 🔑 Step 2 — HuggingFace Login

**Before running**: Add your token in Kaggle → Add-ons → Secrets → Name: `HF_TOKEN`

Also make sure you accepted the Gemma-4-12B license at:
https://huggingface.co/google/gemma-4-12B

In [ ]:
from huggingface_hub import login
import os

# Try Kaggle secrets first, then environment variable
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Token loaded from Kaggle secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("✅ Token loaded from environment")
    else:
        raise ValueError(
            "❌ HF_TOKEN not found!\n"
            "Add it via: Kaggle → Add-ons → Secrets → HF_TOKEN"
        )

login(token=HF_TOKEN, add_to_git_credential=False)
print("✅ Logged into HuggingFace")

## 📁 Step 3 — Clone Lasmoid Repo & Setup Paths

In [ ]:
import subprocess, os, sys

REPO_URL = "https://github.com/Theory903/Lasmoid.git"
REPO_DIR = "/kaggle/working/Lasmoid"

if not os.path.exists(REPO_DIR):
    print("Cloning Lasmoid...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "inference"))
sys.path.insert(0, os.path.join(REPO_DIR, "train"))

print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Files: {os.listdir('.')}")

## ⚙️ Step 4 — Distillation Configuration

Tune these settings based on your Kaggle session time budget.

In [ ]:
# ═══════════════════════════════════════════════════════════
# DISTILLATION CONFIG
# ═══════════════════════════════════════════════════════════

# Teacher
TEACHER_MODEL    = "google/gemma-4-12B"   # 12B dense, 262K vocab
TEACHER_VOCAB    = 262144                  # Gemma-4 vocabulary size

# Student
STUDENT_CONFIG   = "config_100m.json"      # 100M params (~6GB T4 safe)
STUDENT_VOCAB    = 129286                  # Lasmoid BPE tokenizer size
HF_UPLOAD_REPO   = "Theory903/lasmoid-100m-distilled-gemma4-12B"

# Training hyperparams
DISTILL_STEPS    = 5000      # ~15 hrs on T4; increase if you have more time
BATCH_SIZE       = 2         # Micro-batch per grad_accum step
SEQ_LEN          = 256       # Tokens per sequence (keep ≤256 for T4 memory)
GRAD_ACCUM       = 8         # Effective batch = 2 × 8 = 16 sequences

# Distillation loss weights (progressive: start heavy on teacher, end on hard labels)
ALPHA_START      = 0.85      # Weight on teacher soft labels at step 0
ALPHA_END        = 0.25      # Weight on teacher soft labels at final step
TEMPERATURE      = 2.0       # Softening temperature for KL divergence

# Optimizer
LR_MUON          = 2e-3      # Muon LR for 2D weight matrices
LR_ADAMW         = 3e-4      # AdamW LR for embeddings, norms, biases

# Checkpointing
CKPT_DIR         = "/kaggle/working/checkpoints"
LOG_EVERY        = 10
SAVE_EVERY       = 500

# Loss blend: distillation vs student-internal auxiliary losses
DISTILL_WEIGHT   = 0.90
AUX_WEIGHT       = 0.10

import os
os.makedirs(CKPT_DIR, exist_ok=True)

print("✅ Configuration set")
print(f"   Teacher: {TEACHER_MODEL} (262K vocab, 4-bit NF4)")
print(f"   Student: Lasmoid-100M ({STUDENT_VOCAB} vocab)")
print(f"   Steps: {DISTILL_STEPS} | Eff. batch: {BATCH_SIZE*GRAD_ACCUM} | Seq: {SEQ_LEN}")
print(f"   Tokens total: ~{DISTILL_STEPS*BATCH_SIZE*GRAD_ACCUM*SEQ_LEN/1e6:.0f}M")
print(f"   Alpha: {ALPHA_START} → {ALPHA_END} | Temp: {TEMPERATURE}")

## 🏫 Step 5 — Load Teacher: Gemma-4-12B (4-bit NF4)

Loading in **NF4 4-bit** quantization:
- 12B params in BF16 = ~24 GB → in 4-bit NF4 = ~6.5 GB ✅
- Double quantization saves another ~0.3 GB
- Teacher runs frozen (no gradients)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig
import gc

print("📥 Loading Gemma-4-12B teacher in 4-bit NF4...")
print("   First run: downloads ~7GB → ~10-15 min")
print("   Subsequent runs: loads from cache → ~2 min")

# 4-bit NF4 quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NF4 is best for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,  # Compute in bf16 for speed
    bnb_4bit_use_double_quant=True,      # Saves extra ~0.3GB
)

# Load tokenizer
teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
print(f"✅ Teacher tokenizer: vocab_size={len(teacher_tokenizer)}")

# Load model
teacher = AutoModelForImageTextToText.from_pretrained(
    TEACHER_MODEL,
    quantization_config=bnb_config,
    device_map=TEACHER_DEVICE,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",  # More compatible than flash_attn for 4-bit
)

# Freeze teacher completely
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

vram_after = torch.cuda.memory_allocated(0) / 1e9
print(f"\n✅ Teacher loaded!")
print(f"   VRAM used on GPU 0: {vram_after:.2f} GB")
print(f"   Teacher vocab: {teacher.config.vocab_size}")
print(f"   Teacher layers: {teacher.config.num_hidden_layers}")

## 🏗️ Step 6 — Build Student: Lasmoid-100M

In [ ]:
import json
from dataclasses import fields as dc_fields
from transformers import PreTrainedTokenizerFast

# Import Lasmoid model
from inference.model import Lasmoid, ModelArgs

print("🏗️ Building Lasmoid-100M student...")

# Load config
with open(STUDENT_CONFIG) as f:
    cfg = json.load(f)

# Filter to valid ModelArgs fields
valid_fields = {f.name for f in dc_fields(ModelArgs)}
args = ModelArgs(**{k: v for k, v in cfg.items() if k in valid_fields})

# Load student's own BPE tokenizer
student_enc = PreTrainedTokenizerFast.from_pretrained(".")
args.vocab_size = max(args.vocab_size, len(student_enc))
EOS_ID = student_enc.eos_token_id or 1
PAD_ID = student_enc.pad_token_id or EOS_ID

print(f"   Student vocab: {args.vocab_size} tokens")
print(f"   EOS token: {EOS_ID}")

# Initialize student on its device
torch.manual_seed(42)
student = Lasmoid(args).to(STUDENT_DEVICE)
student.train()

n_params = sum(p.numel() for p in student.parameters()) / 1e6
n_trainable = sum(p.numel() for p in student.parameters() if p.requires_grad) / 1e6

if torch.cuda.device_count() > 1:
    vram_student = torch.cuda.memory_allocated(1) / 1e9
else:
    vram_student = torch.cuda.memory_allocated(0) / 1e9

print(f"\n✅ Student built!")
print(f"   Total params: {n_params:.1f}M")
print(f"   Trainable params: {n_trainable:.1f}M")
print(f"   Device: {STUDENT_DEVICE}")
print(f"   VRAM used: {vram_student:.2f} GB")
print(f"   Architecture: dim={args.dim}, layers={args.n_layers}, heads={args.n_heads}")
print(f"   MoE: {args.n_routed_experts} experts, {args.n_activated_experts} active")

## 📚 Step 7 — Load Training Data (FineWeb-Edu Streaming)

In [ ]:
from datasets import load_dataset
import torch

print("📥 Loading FineWeb-Edu dataset (streaming)...")
dataset = load_dataset(
    "HuggingFaceFW/fineweb-edu-score-2",
    split="train",
    streaming=True,
    trust_remote_code=True,
)
dataset_iter = iter(dataset)

def tokenize_buffer(source_iter, n_docs=3000, max_tokens=300_000):
    """Tokenize n_docs from the iterator and pack into SEQ_LEN chunks."""
    tokens = []
    docs_seen = 0
    for doc in source_iter:
        text = doc.get("text", "")
        if text and len(text) > 100:
            ids = student_enc.encode(text, add_special_tokens=False) + [EOS_ID]
            tokens.extend(ids)
            docs_seen += 1
            if docs_seen >= n_docs or len(tokens) >= max_tokens:
                break
    # Pack into sequences
    n = len(tokens) - len(tokens) % SEQ_LEN
    tokens = tokens[:n]
    return torch.tensor(tokens, dtype=torch.long).reshape(-1, SEQ_LEN)

print("Tokenizing initial buffer (~2-3 min)...")
data_buffer = tokenize_buffer(dataset_iter, n_docs=3000)
print(f"✅ Buffer ready: {data_buffer.shape[0]} sequences × {SEQ_LEN} tokens")
print(f"   Total tokens in buffer: {data_buffer.numel():,}")

def get_batch():
    """Sample a random mini-batch from the buffer."""
    idx = torch.randint(data_buffer.shape[0], (BATCH_SIZE,))
    x = data_buffer[idx]  # [B, S] — will be moved to device in the loop
    y = torch.cat([
        x[:, 1:],
        torch.full((x.shape[0], 1), EOS_ID, dtype=torch.long)
    ], dim=1)
    return x, y

# Verify batch
xb, yb = get_batch()
print(f"\nBatch shape: x={xb.shape}, y={yb.shape}")

## 🔧 Step 8 — Setup Optimizers & Loss Functions

In [ ]:
import math
import torch.nn.functional as F
from train.optimizer import build_optimizers
from train.scheduler import WSDScheduler

# ── Optimizers (Muon for 2D weights, AdamW for rest) ────────
opts = build_optimizers(
    student,
    muon_lr=LR_MUON,
    adamw_lr=LR_ADAMW,
    weight_decay=0.1,
)

# ── WSD Learning Rate Schedule ───────────────────────────────
warmup_steps = int(0.02 * DISTILL_STEPS)
stable_steps  = int(0.80 * DISTILL_STEPS)
decay_steps   = DISTILL_STEPS - warmup_steps - stable_steps

scheduler = WSDScheduler(
    opts,
    warmup_steps=warmup_steps,
    stable_steps=stable_steps,
    decay_steps=decay_steps,
    base_lrs=[[g["lr"] for g in o.param_groups] for o in opts],
    min_lr_ratio=0.1,
)

# ── Alpha schedule: teacher weight decays over training ──────
def get_alpha(step: int) -> float:
    """Cosine decay: ALPHA_START → ALPHA_END over DISTILL_STEPS."""
    progress = step / max(DISTILL_STEPS, 1)
    cos = (1 + math.cos(math.pi * progress)) / 2
    return ALPHA_END + (ALPHA_START - ALPHA_END) * cos

# ── Distillation Loss ────────────────────────────────────────
def distillation_loss(
    s_logits: torch.Tensor,   # [B, S, student_vocab]
    t_logits: torch.Tensor,   # [B, S, student_vocab]  (teacher, already sliced)
    labels:   torch.Tensor,   # [B, S]
    alpha:    float,
    T:        float,
    ignore_index: int = -100,
):
    """
    Combined knowledge distillation loss:
    L = (1 - α) * CE(student, hard_labels) + α * T² * KL(teacher_soft || student_soft)
    """
    # Ensure same device
    t_logits = t_logits.to(s_logits.device)
    labels   = labels.to(s_logits.device)

    # Hard label cross-entropy loss
    ce_loss = F.cross_entropy(
        s_logits.reshape(-1, s_logits.size(-1)),
        labels.reshape(-1),
        ignore_index=ignore_index,
    )

    # Soft label KL divergence (forward KL: minimize student from teacher)
    s_log_soft = F.log_softmax(s_logits / T, dim=-1)   # Student: log probs
    t_soft     = F.softmax(t_logits / T, dim=-1)        # Teacher: probs

    # Only compute KL on valid (non-padding) positions
    mask = (labels != ignore_index).unsqueeze(-1).float().to(s_logits.device)
    kl_per_tok = F.kl_div(s_log_soft, t_soft, reduction="none")  # [B, S, V]
    kl_loss = (kl_per_tok.sum(-1, keepdim=True) * mask).sum() / mask.sum().clamp(min=1)

    # Combined loss
    total = (1 - alpha) * ce_loss + alpha * (T ** 2) * kl_loss
    return total, ce_loss, kl_loss

print("✅ Optimizers ready")
print(f"   Muon params: {sum(p.numel() for g in opts[0].param_groups for p in g['params'])/1e6:.1f}M")
print(f"   AdamW params: {sum(p.numel() for g in opts[1].param_groups for p in g['params'])/1e6:.1f}M")
print(f"   LR schedule: warmup={warmup_steps}, stable={stable_steps}, decay={decay_steps}")

## 🚀 Step 9 — DISTILLATION TRAINING LOOP

This is the main training loop. It will run for `DISTILL_STEPS` steps and:
1. Feed each batch to **Gemma-4-12B** (teacher) → get soft probability distributions
2. Feed same batch to **Lasmoid-100M** (student) → get its predictions
3. Train student to **match teacher's distributions** via KL divergence
4. Also include hard-label cross-entropy for grounding
5. Include student's own auxiliary losses (MoE routing, VQ, etc.)

In [ ]:
import time
import gc
from tqdm.notebook import tqdm

# ── Import student's auxiliary loss function ──────────────────
from inference.model import compute_loss as student_compute_loss

print("╔══════════════════════════════════════════════════════════╗")
print("║  🧠 Gemma-4-12B → Lasmoid-100M Distillation START       ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Steps: {DISTILL_STEPS:<8} Eff.Batch: {BATCH_SIZE*GRAD_ACCUM:<6} Temp: {TEMPERATURE}        ║")
print(f"║  Alpha: {ALPHA_START}→{ALPHA_END}   LR(Muon): {LR_MUON}  LR(Adam): {LR_ADAMW}  ║")
print("╚══════════════════════════════════════════════════════════╝")

# ── Training state ────────────────────────────────────────────
history = {"loss": [], "ce": [], "kl": [], "alpha": []}
t0 = time.time()
best_loss = float("inf")

pbar = tqdm(range(DISTILL_STEPS), desc="Distilling", dynamic_ncols=True)

for step in pbar:
    # Update LR schedule
    scheduler.step(step)
    alpha = get_alpha(step)

    # Zero gradients
    for opt in opts:
        opt.zero_grad(set_to_none=True)

    # Gradient accumulation
    step_loss = step_ce = step_kl = 0.0

    for acc_step in range(GRAD_ACCUM):
        x, y = get_batch()
        x_teacher = x.to(TEACHER_DEVICE)  # Teacher input
        x_student = x.to(STUDENT_DEVICE)  # Student input
        y_student = y.to(STUDENT_DEVICE)  # Labels on student device

        # ── Teacher Forward (frozen, 4-bit, no grad) ──────────
        with torch.no_grad():
            t_out = teacher(input_ids=x_teacher)
            # Gemma-4 has 262K vocab; slice to student vocab size
            # (shared BPE prefix: first 129K tokens largely overlap)
            t_logits = t_out.logits[:, :, :STUDENT_VOCAB].float()  # [B, S, 129286]
            # Move to student device for loss computation
            t_logits = t_logits.to(STUDENT_DEVICE)

        # ── Student Forward (bf16, full precision for grads) ──
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            # Student returns: (logits, mtp_logits, ..., route_maps, ..., adj, expert_probs)
            s_out = student(x_student, x_student)

            # Unpack student outputs
            if isinstance(s_out, tuple):
                s_logits = s_out[0]   # Primary logits [B, S, vocab]
                # Auxiliary outputs for student's internal losses
                mtp_out  = s_out[1] if len(s_out) > 1 else None
                rmaps    = s_out[4] if len(s_out) > 4 else []
                adjs     = s_out[6] if len(s_out) > 6 else []
                eprobs   = s_out[7] if len(s_out) > 7 else []
            else:
                s_logits = s_out
                rmaps = adjs = eprobs = []

            # ── Primary distillation loss (KL + CE) ──────────
            d_loss, ce_loss, kl_loss = distillation_loss(
                s_logits, t_logits, y_student,
                alpha=alpha, T=TEMPERATURE
            )

            # ── Student auxiliary losses (MoE, VQ, etc.) ─────
            try:
                loss_mask = torch.ones_like(x_student, dtype=torch.float32)
                aux_loss = student_compute_loss(
                    s_logits, y_student,
                    rmaps,
                    [student.last_vq_loss] if hasattr(student, 'last_vq_loss') else [],
                    adjs, eprobs,
                    loss_mask=loss_mask,
                    moe_aux_loss=student.last_moe_loss if hasattr(student, 'last_moe_loss') else 0.0,
                    ignore_index=-100,
                )
            except Exception:
                aux_loss = d_loss * 0.0  # Skip aux if unavailable

            # ── Blend: 90% distillation + 10% auxiliary ──────
            total_loss = DISTILL_WEIGHT * d_loss + AUX_WEIGHT * aux_loss
            total_loss = total_loss / GRAD_ACCUM  # Scale for accumulation

        # Backward pass
        total_loss.backward()

        step_loss += total_loss.item() * GRAD_ACCUM
        step_ce   += ce_loss.item()
        step_kl   += kl_loss.item()

    # ── Gradient clipping + optimizer step ───────────────────
    torch.nn.utils.clip_grad_norm_(student.parameters(), max_norm=1.0)
    for opt in opts:
        opt.step()

    # ── Track metrics ─────────────────────────────────────────
    history["loss"].append(step_loss)
    history["ce"].append(step_ce / GRAD_ACCUM)
    history["kl"].append(step_kl / GRAD_ACCUM)
    history["alpha"].append(alpha)

    # ── Progress bar update ───────────────────────────────────
    if step % LOG_EVERY == 0:
        elapsed = time.time() - t0
        tok_per_s = (step + 1) * BATCH_SIZE * GRAD_ACCUM * SEQ_LEN / max(elapsed, 1)
        avg_loss = sum(history["loss"][-LOG_EVERY:]) / min(len(history["loss"]), LOG_EVERY)
        eta_h = (DISTILL_STEPS - step) / max(step + 1, 1) * elapsed / 3600

        pbar.set_postfix({
            "loss":   f"{avg_loss:.3f}",
            "ce":     f"{step_ce/GRAD_ACCUM:.3f}",
            "kl":     f"{step_kl/GRAD_ACCUM:.3f}",
            "α":      f"{alpha:.2f}",
            "tok/s":  f"{tok_per_s:.0f}",
            "ETA":    f"{eta_h:.1f}h",
        })

        # Also print detailed log every 100 steps
        if step % 100 == 0:
            gpu0_mem = torch.cuda.memory_allocated(0) / 1e9
            gpu1_mem = torch.cuda.memory_allocated(1) / 1e9 if torch.cuda.device_count() > 1 else 0
            print(f"\nStep {step:5d}/{DISTILL_STEPS} | "
                  f"loss={avg_loss:.3f} | ce={step_ce/GRAD_ACCUM:.3f} | "
                  f"kl={step_kl/GRAD_ACCUM:.3f} | α={alpha:.2f} | "
                  f"GPU0={gpu0_mem:.1f}GB | GPU1={gpu1_mem:.1f}GB | "
                  f"ETA={eta_h:.1f}h")

    # ── Save checkpoint ───────────────────────────────────────
    if step > 0 and step % SAVE_EVERY == 0:
        avg_recent = sum(history["loss"][-SAVE_EVERY:]) / SAVE_EVERY
        ckpt_path = f"{CKPT_DIR}/distill_step_{step:05d}.pt"
        torch.save({
            "step":             step,
            "model_state_dict": student.state_dict(),
            "optimizer_states": [o.state_dict() for o in opts],
            "loss_history":     history["loss"][-SAVE_EVERY:],
            "alpha":            alpha,
            "config":           cfg,
        }, ckpt_path)
        is_best = avg_recent < best_loss
        if is_best:
            best_loss = avg_recent
            torch.save(student.state_dict(), f"{CKPT_DIR}/best_model.pt")
        print(f"\n💾 Checkpoint: step={step}, loss={avg_recent:.4f}"
              f"{' ← BEST' if is_best else ''}")

    # ── Refill data buffer every 1000 steps ──────────────────
    if step > 0 and step % 1000 == 0:
        print(f"\n🔄 Refilling data buffer at step {step}...")
        data_buffer = tokenize_buffer(dataset_iter, n_docs=3000)
        print(f"   New buffer: {data_buffer.shape[0]} sequences")
        gc.collect()
        torch.cuda.empty_cache()

# ── Training complete ─────────────────────────────────────────
elapsed_h = (time.time() - t0) / 3600
final_loss = sum(history["loss"][-100:]) / min(len(history["loss"]), 100)
print(f"\n{'='*60}")
print(f"✅ DISTILLATION COMPLETE")
print(f"   Time: {elapsed_h:.1f} hours")
print(f"   Loss: {history['loss'][0]:.3f} → {final_loss:.3f}")
print(f"   Best loss: {best_loss:.3f}")
print(f"{'='*60}")

## 💾 Step 10 — Save Final Model

In [ ]:
import torch

# Save final checkpoint
final_path = f"{CKPT_DIR}/distill_final.pt"
torch.save({
    "step":             DISTILL_STEPS,
    "model_state_dict": student.state_dict(),
    "loss_history":     history["loss"],
    "config":           cfg,
    "teacher":          TEACHER_MODEL,
    "distill_steps":    DISTILL_STEPS,
}, final_path)

print(f"✅ Final model saved: {final_path}")
print(f"   Size: {os.path.getsize(final_path)/1e6:.1f} MB")

## ☁️ Step 11 — Upload to HuggingFace Hub

In [ ]:
import shutil
from safetensors.torch import save_file
from huggingface_hub import HfApi, create_repo

print(f"☁️  Uploading to {HF_UPLOAD_REPO}...")

# Prepare upload directory
upload_dir = f"{CKPT_DIR}/hf_upload"
os.makedirs(upload_dir, exist_ok=True)

# Save model weights in safetensors format
save_file(student.state_dict(), f"{upload_dir}/model.safetensors")

# Copy config and tokenizer files
shutil.copy(STUDENT_CONFIG, f"{upload_dir}/config.json")
for fname in ["tokenizer.json", "tokenizer_config.json", "generation_config.json"]:
    if os.path.exists(fname):
        shutil.copy(fname, upload_dir)

# Write README
elapsed_h = (time.time() - t0) / 3600
with open(f"{upload_dir}/README.md", "w") as f:
    f.write(f"""---
license: apache-2.0
tags:
  - lasmoid
  - distillation
  - gemma4
  - moe
  - mamba
  - hybrid-transformer
base_model: google/gemma-4-12B
---

# Lasmoid-100M (Distilled from Gemma-4-12B)

100M parameter Lasmoid Hybrid Concept Transformer distilled from Google's Gemma-4-12B.

## Training Details

| Property | Value |
|----------|-------|
| **Teacher** | google/gemma-4-12B (12B params) |
| **Teacher Quantization** | 4-bit NF4 (bitsandbytes) |
| **Student** | Lasmoid-100M |
| **Method** | Online KL-divergence distillation |
| **Dataset** | FineWeb-Edu (streaming) |
| **Steps** | {DISTILL_STEPS:,} |
| **Effective Batch** | {BATCH_SIZE * GRAD_ACCUM} × {SEQ_LEN} tokens |
| **Temperature** | {TEMPERATURE} |
| **Alpha schedule** | {ALPHA_START} → {ALPHA_END} (cosine) |
| **Platform** | Kaggle T4 x2 (free) |
| **Training time** | {elapsed_h:.1f} hours |

## Architecture

Lasmoid is a hybrid transformer featuring:
- **CSA** (Compressed Sparse Attention) + **Mamba-2 SSM** interleaved layers
- **Grey-Box MoE** ({args.n_routed_experts} routed + {args.n_shared_experts} shared experts)
- **Manifold-Constrained Hyper-Connections** (mHC residual routing)
- **Concept Memory** (VQ codebook with {args.codebook_size} codes)
- **MTP** (Multi-Token Prediction) heads

## Loss

Final training loss: {final_loss:.4f}  
Best checkpoint loss: {best_loss:.4f}
""")

# Create repo and upload
api = HfApi()
create_repo(HF_UPLOAD_REPO, exist_ok=True, private=False)
api.upload_folder(
    folder_path=upload_dir,
    repo_id=HF_UPLOAD_REPO,
    commit_message=f"Distilled from Gemma-4-12B | {DISTILL_STEPS} steps | loss={final_loss:.4f}",
)

print(f"\n✅ Uploaded successfully!")
print(f"   🔗 https://huggingface.co/{HF_UPLOAD_REPO}")

## 🧪 Step 12 — Quick Inference Test

In [ ]:
import torch

student.eval()

test_prompts = [
    "The principles of quantum mechanics explain",
    "Machine learning models are trained by",
    "To be, or not to be,",
]

print("🧪 Testing distilled student model...\n")

for prompt in test_prompts:
    ids = student_enc.encode(prompt, add_special_tokens=False)
    x = torch.tensor([ids], dtype=torch.long).to(STUDENT_DEVICE)

    with torch.no_grad():
        generated = list(ids)
        for _ in range(40):
            inp = torch.tensor([generated[-SEQ_LEN:]], dtype=torch.long).to(STUDENT_DEVICE)
            out = student(inp, inp)
            logits = out[0] if isinstance(out, tuple) else out
            next_tok = logits[0, -1].argmax().item()
            if next_tok == EOS_ID:
                break
            generated.append(next_tok)

    completion = student_enc.decode(generated)
    print(f"PROMPT : {prompt}")
    print(f"OUTPUT : {completion}")
    print("-" * 60)

student.train()

## 📊 Step 13 — Plot Training Curves

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Kaggle
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Gemma-4-12B → Lasmoid-100M Distillation", fontsize=14, fontweight='bold')

# Smooth with rolling average
def smooth(vals, window=50):
    out = []
    for i in range(len(vals)):
        start = max(0, i - window)
        out.append(sum(vals[start:i+1]) / (i - start + 1))
    return out

steps = list(range(len(history["loss"])))

axes[0].plot(steps, smooth(history["loss"]), color="#6366f1", linewidth=1.5)
axes[0].set_title("Total Loss")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(steps, smooth(history["ce"]), color="#10b981", linewidth=1.5, label="CE")
axes[1].plot(steps, smooth(history["kl"]), color="#f59e0b", linewidth=1.5, label="KL")
axes[1].set_title("CE vs KL Loss")
axes[1].set_xlabel("Step")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(steps, history["alpha"], color="#ec4899", linewidth=1.5)
axes[2].set_title("Alpha Schedule (Teacher Weight)")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Alpha")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = f"{CKPT_DIR}/training_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Plot saved: {plot_path}")